# L20: 向量检索技术

> 深入理解 RAG 系统中的检索机制

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env.file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.rag.retriever import Retriever
print("✓ 模块导入成功")

## 20.1 向量检索概述

### 检索流程

```
1. 用户提问
   ↓
2. 将问题转化为向量 (Embedding)
   ↓
3. 在向量库中搜索最相似的文档
   ↓
4. 返回 Top-K 个最相关的文档
```

### 核心概念

| 概念 | 说明 |
|------|------|
| **查询向量** | 用户问题的 Embedding |
| **相似度计算** | 计算查询向量和文档向量的相似度 |
| **Top-K** | 返回最相似的前 K 个结果 |
| **阈值过滤** | 只返回相似度超过阈值的结果 |

## 20.2 相似度计算方法

### 常见相似度度量

In [ ]:
import numpy as np

def cosine_similarity(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    余弦相似度
    值域: [-1, 1], 越接近 1 越相似
    """
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def euclidean_distance(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    欧氏距离
    值域: [0, +∞), 越接近 0 越相似
    """
    return np.linalg.norm(vec1 - vec2)

def dot_product(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    点积
    适用于已归一化的向量
    """
    return np.dot(vec1, vec2)

# 演示相似度计算
def demo_similarity_metrics():
    print("=== 相似度计算方法对比 ===\n")
    
    # 测试向量
    query = np.array([0.9, 0.1, 0.8])
    doc1 = np.array([0.8, 0.2, 0.7])  # 相似
    doc2 = np.array([0.1, 0.9, 0.2])  # 不相似
    
    print("查询向量:", query)
    print("文档 1 (相似):", doc1)
    print("文档 2 (不相似):", doc2)
    print()
    
    # 计算相似度
    print("余弦相似度:")
    print(f"  查询 vs 文档1: {cosine_similarity(query, doc1):.4f}")
    print(f"  查询 vs 文档2: {cosine_similarity(query, doc2):.4f}")
    print()
    
    print("欧氏距离:")
    print(f"  查询 vs 文档1: {euclidean_distance(query, doc1):.4f}")
    print(f"  查询 vs 文档2: {euclidean_distance(query, doc2):.4f}")
    print()
    
    print("点积 (归一化向量):")
    query_norm = query / np.linalg.norm(query)
    doc1_norm = doc1 / np.linalg.norm(doc1)
    doc2_norm = doc2 / np.linalg.norm(doc2)
    print(f"  查询 vs 文档1: {dot_product(query_norm, doc1_norm):.4f}")
    print(f"  查询 vs 文档2: {dot_product(query_norm, doc2_norm):.4f}")

demo_similarity_metrics()

## 20.3 简单向量检索器实现

In [ ]:
class SimpleVectorRetriever:
    """
    简单的向量检索器
    """
    
    def __init__(self, documents: List[Dict[str, Any]] = None):
        """
        初始化检索器
        
        Args:
            documents: 文档列表，每个文档包含 content 和 embedding
        """
        self.documents = documents or []
    
    def add_document(self, content: str, embedding: List[float], metadata: Dict = None):
        """添加文档"""
        self.documents.append({
            "content": content,
            "embedding": np.array(embedding),
            "metadata": metadata or {}
        })
    
    def retrieve(
        self,
        query_embedding: List[float],
        top_k: int = 5,
        score_threshold: float = None
    ) -> List[Dict[str, Any]]:
        """
        检索最相似的文档
        
        Args:
            query_embedding: 查询向量
            top_k: 返回前 K 个结果
            score_threshold: 相似度阈值，低于此值的结果不返回
            
        Returns:
            检索结果列表，按相似度排序
        """
        if not self.documents:
            return []
        
        query_vec = np.array(query_embedding)
        
        # 计算相似度
        results = []
        for doc in self.documents:
            similarity = cosine_similarity(query_vec, doc["embedding"])
            
            # 应用阈值过滤
            if score_threshold is None or similarity >= score_threshold:
                results.append({
                    "content": doc["content"],
                    "score": similarity,
                    "metadata": doc["metadata"]
                })
        
        # 按相似度排序
        results.sort(key=lambda x: x["score"], reverse=True)
        
        return results[:top_k]

# 演示检索
def demo_retrieval():
    print("=== 向量检索演示 ===\n")
    
    # 创建检索器
    retriever = SimpleVectorRetriever()
    
    # 添加文档（使用 mock embedding）
    docs = [
        ("Python 是一种编程语言", [0.9, 0.1, 0.8]),
        ("Java 也是流行的编程语言", [0.8, 0.2, 0.7]),
        ("今天天气很好", [0.1, 0.9, 0.2]),
        ("我喜欢吃苹果", [0.2, 0.8, 0.1]),
    ]
    
    for content, emb in docs:
        retriever.add_document(content, emb)
    
    print("已添加 4 个文档\n")
    
    # 查询
    query_emb = [0.85, 0.15, 0.75]  # 接近编程语言
    results = retriever.retrieve(query_emb, top_k=3)
    
    print("查询: 编程语言相关")
    print("\n检索结果:")
    for i, r in enumerate(results, 1):
        print(f"  {i}. {r['content']}")
        print(f"     相似度: {r['score']:.4f}")

demo_retrieval()

## 20.4 高级检索技巧

### 混合检索 (Hybrid Search)

结合向量检索和关键词检索

In [ ]:
class HybridRetriever:
    """
    混合检索器 - 向量检索 + 关键词检索
    """
    
    def __init__(self, vector_retriever: SimpleVectorRetriever):
        self.vector_retriever = vector_retriever
    
    def retrieve(
        self,
        query_text: str,
        query_embedding: List[float],
        top_k: int = 5,
        alpha: float = 0.7  # 向量检索权重
    ) -> List[Dict[str, Any]]:
        """
        混合检索
        
        Args:
            query_text: 查询文本（用于关键词匹配）
            query_embedding: 查询向量（用于向量检索）
            top_k: 返回结果数
            alpha: 向量检索权重（0-1），关键词检索权重为 1-alpha
        """
        # 1. 向量检索
        vector_results = self.vector_retriever.retrieve(query_embedding, top_k=top_k*2)
        
        # 2. 计算关键词匹配分数
        query_keywords = set(query_text.lower().split())
        
        # 合并结果
        combined_scores = {}
        
        for r in vector_results:
            content = r["content"]
            content_lower = content.lower()
            
            # 关键词分数：匹配的关键词数 / 查询关键词数
            matched_keywords = sum(1 for kw in query_keywords if kw in content_lower)
            keyword_score = matched_keywords / len(query_keywords) if query_keywords else 0
            
            # 混合分数
            combined_score = alpha * r["score"] + (1 - alpha) * keyword_score
            
            combined_scores[content] = {
                "content": content,
                "vector_score": r["score"],
                "keyword_score": keyword_score,
                "combined_score": combined_score,
                "metadata": r["metadata"]
            }
        
        # 按混合分数排序
        sorted_results = sorted(
            combined_scores.values(),
            key=lambda x: x["combined_score"],
            reverse=True
        )
        
        return sorted_results[:top_k]

# 演示混合检索
def demo_hybrid_retrieval():
    print("=== 混合检索演示 ===\n")
    
    # 创建检索器
    vector_retriever = SimpleVectorRetriever()
    hybrid_retriever = HybridRetriever(vector_retriever)
    
    # 添加文档
    docs = [
        ("Python 编程语言教程", [0.9, 0.1, 0.8]),
        ("Java 开发指南", [0.8, 0.2, 0.7]),
        ("Python 数据分析", [0.85, 0.15, 0.75]),
    ]
    
    for content, emb in docs:
        vector_retriever.add_document(content, emb)
    
    # 查询
    query_text = "Python 编程"
    query_emb = [0.88, 0.12, 0.78]
    
    results = hybrid_retriever.retrieve(query_text, query_emb, top_k=3, alpha=0.5)
    
    print(f"查询: {query_text}\n")
    print("检索结果:")
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['content']}")
        print(f"   向量分数: {r['vector_score']:.3f}")
        print(f"   关键词分数: {r['keyword_score']:.3f}")
        print(f"   混合分数: {r['combined_score']:.3f}")
        print()

demo_hybrid_retrieval()

### 重排序 (Reranking)

对检索结果进行重新排序以提高质量

In [ ]:
class Reranker:
    """
    结果重排序器
    """
    
    def __init__(self, method: str = "score"):
        self.method = method
    
    def rerank(
        self,
        results: List[Dict[str, Any]],
        query: str
    ) -> List[Dict[str, Any]]:
        """
        对检索结果重新排序
        
        Args:
            results: 原始检索结果
            query: 查询文本
            
        Returns:
            重排序后的结果
        """
        if self.method == "score":
            # 已经按分数排序，不需要重排
            return results
        
        elif self.method == "keyword_boost":
            # 提升包含查询关键词的结果
            query_lower = query.lower()
            query_words = set(query_lower.split())
            
            for r in results:
                content_lower = r["content"].lower()
                # 计算匹配的查询词数量
                matches = sum(1 for w in query_words if w in content_lower)
                # 提升分数
                r["rerank_score"] = r["score"] * (1 + matches * 0.1)
            
            # 按重排分数排序
            results.sort(key=lambda x: x.get("rerank_score", x["score"]), reverse=True)
            
            return results
        
        return results

# 演示重排序
def demo_reranking():
    print("=== 重排序演示 ===\n")
    
    # 模拟检索结果
    results = [
        {"content": "Python 语言基础", "score": 0.85},
        {"content": "编程入门教程", "score": 0.82},
        {"content": "Python 高级技巧", "score": 0.78},
        {"content": "数据分析方法", "score": 0.75},
    ]
    
    print("原始结果:")
    for i, r in enumerate(results, 1):
        print(f"{i}. {r['content']} (分数: {r['score']:.2f})")
    
    print("\n重排序后 (关键词提升):")
    reranker = Reranker(method="keyword_boost")
    reranked = reranker.rerank(results.copy(), "Python 编程")
    
    for i, r in enumerate(reranked, 1):
        rerank_score = r.get("rerank_score", r["score"])
        print(f"{i}. {r['content']} (分数: {r['score']:.2f} → {rerank_score:.2f})")

demo_reranking()

## 20.5 检索评估指标

### 关键指标

In [ ]:
class RetrievalMetrics:
    """
    检索评估指标
    """
    
    @staticmethod
    def precision(relevant: List, retrieved: List) -> float:
        """
        精确率 = 检索出的相关文档数 / 检索出的文档总数
        """
        if not retrieved:
            return 0.0
        return len(set(relevant) & set(retrieved)) / len(retrieved)
    
    @staticmethod
    def recall(relevant: List, retrieved: List, total_relevant: int) -> float:
        """
        召回率 = 检索出的相关文档数 / 总相关文档数
        """
        if total_relevant == 0:
            return 0.0
        return len(set(relevant) & set(retrieved)) / total_relevant
    
    @staticmethod
    def mrr(relevant_items: List[str], ranked_results: List[str]) -> float:
        """
        平均倒数排名 (Mean Reciprocal Rank)
        关注第一个相关文档的位置
        """
        for i, item in enumerate(ranked_results, 1):
            if item in relevant_items:
                return 1.0 / i
        return 0.0
    
    @staticmethod
    def ndcg(relevant_scores: List[int], k: int = 10) -> float:
        """
        归一化折损累计增益 (NDCG)
        考虑排序位置和相关性等级
        """
        import math
        
        dcg = sum((2**score - 1) / math.log2(i + 2) 
                 for i, score in enumerate(relevant_scores[:k]))
        
        # 理想排序：分数降序排列
        ideal_scores = sorted(relevant_scores, reverse=True)[:k]
        idcg = sum((2**score - 1) / math.log2(i + 2) 
                    for i, score in enumerate(ideal_scores))
        
        return dcg / idcg if idcg > 0 else 0.0

# 演示评估指标
def demo_metrics():
    print("=== 检索评估指标演示 ===\n")
    
    # 假设场景
    all_relevant = ["doc1", "doc3", "doc5", "doc7"]  # 总共相关文档
    retrieved = ["doc1", "doc2", "doc3", "doc4", "doc5"]  # 检索结果
    
    print(f"总相关文档: {all_relevant}")
    print(f"检索结果: {retrieved}\n")
    
    # 计算指标
    precision = RetrievalMetrics.precision(all_relevant, retrieved)
    recall = RetrievalMetrics.recall(all_relevant, retrieved, len(all_relevant))
    mrr = RetrievalMetrics.mrr(all_relevant, retrieved)
    
    # 模拟相关性分数 (0-3)
    relevance_scores = [3, 0, 2, 0, 1]  # 对应 retrieved 的相关性
    ndcg = RetrievalMetrics.ndcg(relevance_scores)
    
    print(f"精确率 (Precision): {precision:.2%}")
    print(f"召回率 (Recall): {recall:.2%}")
    print(f"MRR: {mrr:.3f}")
    print(f"NDCG@5: {ndcg:.3f}")

demo_metrics()

## 20.6 常见面试问题

**Q: Top-K 设置多大合适？**
- A:
  - **太小**（1-2）：可能遗漏相关信息
  - **太大**（10+）：噪音多，Token 消耗大
  - **推荐值**：3-5
  - **动态调整**：根据查询复杂度调整

**Q: 如何处理检索结果质量差？**
- A:
  1. **检查 Embedding 质量**：模型是否匹配内容类型
  2. **调整 Chunk 大小**：太大或太小都会影响质量
  3. **增加 Overlap**：提高信息完整性
  4. **使用混合检索**：向量 + 关键词
  5. **添加重排序**：对结果重新评分

**Q: 向量检索和关键词检索有什么区别？**
- A:
  | 特性 | 向量检索 | 关键词检索 |
  |------|----------|------------|
  | 匹配方式 | 语义相似度 | 精确匹配 |
  | 查询理解 | 理解意图 | 需要精确关键词 |
  | 适用场景 | 自然语言查询 | 专业术语查找 |
  | 实现成本 | 需要 Embedding | 简单 |
  | 推荐策略 | 两者结合 | |

**Q: 如何优化检索速度？**
- A:
  1. **使用索引**：HNSW、IVF 等
  2. **降维**：PCA 等方法减少向量维度
  3. **量化**：减少存储和计算开销
  4. **缓存**：常见查询的缓存结果
  5. **分片**：大规模数据分片检索

---

## 总结

本课程学习了向量检索的核心知识：

| 知识点 | 说明 |
|--------|------|
| **检索流程** | 查询 → Embedding → 相似度计算 → Top-K |
| **相似度度量** | 余弦相似度、欧氏距离、点积 |
| **混合检索** | 向量检索 + 关键词检索 |
| **重排序** | 提升检索结果质量 |
| **评估指标** | Precision、Recall、MRR、NDCG |
| **优化技巧** | 索引、降维、量化、缓存 |

**下一步**: L21 将学习完整的 RAG 流程实现！